# Main Satellite

In [21]:
!pip install skyfield sgp4
import pandas as pd
import numpy as np

## 1) Loading dataset

In [22]:
main_df = pd.read_csv("/content/drive/MyDrive/main satellite.txt")

main_df.head()

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT
0,PRSS-1,2018-056B,2026-05-15T05:46:05.827008,14.819337,0.000186,97.7837,204.0211,94.6085,265.5341,0,U,43530,999,42355,0.000044,0.000003,0


This data is in the OMM (Orbital Mean-Elements Message) format which is more modern than raw TLE data.

## 2) Creating the  main satellite object

In [23]:
from skyfield.api import load, EarthSatellite
from sgp4.api import Satrec

ts = load.timescale()

row = main_df.iloc[0]

In [24]:
satrec = Satrec()
#converting OMM to a satellite object
#basically converting orbital parameters to sgp4 orbital model
satrec.sgp4init(
    0,                        #gravity model
    'i',                      #improved mode
    int(row['NORAD_CAT_ID']),
    0.0,
    float(row['BSTAR']),
    float(row['MEAN_MOTION_DOT']),
    float(row['MEAN_MOTION_DDOT']),
    float(row['ECCENTRICITY']),
    np.radians(float(row['ARG_OF_PERICENTER'])),
    np.radians(float(row['INCLINATION'])),
    np.radians(float(row['MEAN_ANOMALY'])),
    float(row['MEAN_MOTION']) * 2 * np.pi / 1440.0,
    np.radians(float(row['RA_OF_ASC_NODE']))
)

In [25]:
#creating a skyfield satellite
main_sat = EarthSatellite.from_satrec(
    satrec,
    ts
)

In [26]:
t = ts.now()

pos = main_sat.at(t).position.km

print(pos)

[-4377.75486463 -2086.64926134  4928.75000722]


We got the coordinates! Yay

## 3) Creating main satellite trajectory

In [27]:
#creating 60 minutes timestamps
times = ts.utc(2026, 5, 1, range(0, 60))

main_positions = []

for t in times:

    pos = main_sat.at(t).position.km

    main_positions.append({
        "time": str(t),
        "x": pos[0],
        "y": pos[1],
        "z": pos[2]
    })
main_df = pd.DataFrame(main_positions)

# Collision Module

In [28]:
traj_df = pd.read_csv("/content/drive/MyDrive/trajectories.csv")

traj_df.head()

,object_id,time,x,y,z
0,FY1C_001,<Time tt=2461161.500800741>,5027.707246,1971.935225,4729.284992
1,FY1C_001,<Time tt=2461161.5424674074>,-6865.862776,-1829.267571,-1025.230159
2,FY1C_001,<Time tt=2461161.584134074>,6400.032857,1059.949403,-3108.278702
3,FY1C_001,<Time tt=2461161.625800741>,-3738.692383,65.963274,6115.213287
4,FY1C_001,<Time tt=2461161.6674674074>,-164.454450,-1175.537838,-7093.128075


In [29]:
collision_risks = []

#change the value of the threshold to see the collision risk at certain distances
threshold = 1000  #km

#main collision detection logic
for idx, main_row in main_df.iterrows():

    current_time = main_row['time']

    debris_at_time = traj_df[
        traj_df['time'] == current_time
    ]

    for _, debris_row in debris_at_time.iterrows():
        #using euclidean distance
        distance = np.sqrt(
            (debris_row['x'] - main_row['x'])**2 +
            (debris_row['y'] - main_row['y'])**2 +
            (debris_row['z'] - main_row['z'])**2
        )

        if distance < threshold:

            collision_risks.append({
                "time": current_time,
                "debris_object": debris_row['object_id'],
                "distance_km": distance
            })

For each timestamp:

- We are getting the main satellite position

- Finding all debris positions at same time

- Computing distances

- Detecting close approaches

In [30]:
collision_df = pd.DataFrame(collision_risks)

collision_df.head()

,time,debris_object,distance_km
0,<Time tt=2461162.125800741>,FY1C_006,875.314201
